# Week 14 — Cross-Validation

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week14/`)
- `biz_customers.csv` — ลูกค้า 1203 คน (ทุกข้อใช้ทั้งไฟล์)

## กฎของสัปดาห์นี้ — อ่านให้ดี

1. **ไม่สับข้อมูลเด็ดขาด** → ใช้ `KFold` / `StratifiedKFold` / `LeaveOneOut`
   **โดยไม่ตั้ง `shuffle`** (ค่าเริ่มต้นคือ False) และห้ามใส่ `random_state`
   ห้ามใช้ `train_test_split`
2. **"MSE ของ fold"** = MSE บน validation fold นั้น
   **"ค่า CV"** = ค่าเฉลี่ยของทุก fold (**ทุก fold น้ำหนักเท่ากัน** ไม่ถ่วงตามขนาด)
3. **standardize ใหม่ในทุก fold** โดยใช้เฉพาะ **train fold** เท่านั้น
   (ถ้าใช้ค่าเฉลี่ยของทั้งไฟล์ = ข้อมูลรั่วไหลจาก validation เข้าไปในการ fit)
   → เขียนเป็น `make_pipeline(StandardScaler(), โมเดล())` แล้ว fit ใหม่ในแต่ละ fold
4. **KNN classifier ใช้เสียงข้างมาก** → ต้องใช้ **K เป็นเลขคี่** ไม่งั้นเกิดการโยงเสมอ
5. `StratifiedKFold` ต้องส่ง **y** เป็นตัวแบ่งชั้น: `split(X, y)`

## วิธี CV ที่ใช้ในสัปดาห์นี้
| วิธี | ตัวแบ่ง | ใช้เมื่อ |
|---|---|---|
| Validation set | เอง (เช่น เลขคู่/เลขคี่) | ต้องการความเร็ว |
| K-Fold | `KFold(n_splits=K)` | ต้องการใช้ข้อมูลทุกแถวเป็น train สักครั้ง |
| Stratified K-Fold | `StratifiedKFold(n_splits=K)` | ต้องรักษา **สัดส่วนคลาส** ให้ใกล้เคียงกันทุก fold |
| Leave-One-Out | `LeaveOneOut()` | ข้อมูลน้อย ต้องการความเป็นอิสระสูงสุด (แต่ช้า) |

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (ทำครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week14')
NEEDED = ['biz_customers.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ดาวน์โหลด: ['biz_customers.csv']


  บันทึก data\week14\biz_customers.csv


In [2]:
# --- ตรวจไฟล์ข้อมูล + เวอร์ชันเครื่องมือ (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd
import sklearn

DATA_DIR = Path(f'data/week14')
NEEDED = ['biz_customers.csv']

print('scikit-learn', sklearn.__version__, '| pandas', pd.__version__)
for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์')
    print('    churn:', df['churn'].value_counts().to_dict())
    print('    responded:', df['responded'].value_counts().to_dict())

scikit-learn 1.9.0 | pandas 3.0.5
biz_customers.csv       1203 แถว x 13 คอลัมน์
    churn: {0: 1060, 1: 143}
    responded: {0: 832, 1: 371}


## ข้อ 14.1 — Stratified 5-Fold CV: ค่าเฉลี่ย F1 ของ KNN (K=11)

- `StratifiedKFold(n_splits=5).split(X, y)` → ต้องส่ง `y` เพื่อให้สัดส่วนคลาสใกล้เคียงกันทุก fold
- **ต้อง fit โมเดลใหม่ในทุก fold** (pipeline จะจัดการเรื่อง standardize ให้ถูกเอง)
- ค่า CV = ค่าเฉลี่ยของ F1 ทั้ง 5 fold

In [3]:
# --- ข้อ 14.1 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cust = pd.read_csv(Path('data/week14') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

cols = ['email_opens', 'income_k', 'basket_size', 'spend_k']
X = cust[cols].to_numpy(dtype=float)
y = cust['responded'].to_numpy(dtype=int)

f1s = []
for i, (tr, va) in enumerate(StratifiedKFold(n_splits=5).split(X, y), start=1):
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=11))
    pipe.fit(X[tr], y[tr])                    # scaler fit ด้วย train fold เท่านั้น
    f1 = f1_score(y[va], pipe.predict(X[va]), pos_label=1)
    f1s.append(f1)
    print(f'  fold {i}: train = {len(tr)} | validation = {len(va)} | '
          f'สัดส่วน responded ใน val = {y[va].mean():.4f} | F1 = {f1:.6f}')

answer = float(np.mean(f1s))

print()
print('ค่าเฉลี่ย 5 fold =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

  fold 1: train = 962 | validation = 241 | สัดส่วน responded ใน val = 0.3071 | F1 = 0.262626
  fold 2: train = 962 | validation = 241 | สัดส่วน responded ใน val = 0.3071 | F1 = 0.283019
  fold 3: train = 962 | validation = 241 | สัดส่วน responded ใน val = 0.3112 | F1 = 0.175439
  fold 4: train = 963 | validation = 240 | สัดส่วน responded ใน val = 0.3083 | F1 = 0.299065
  fold 5: train = 963 | validation = 240 | สัดส่วน responded ใน val = 0.3083 | F1 = 0.252427

ค่าเฉลี่ย 5 fold = 0.2545152664137572
ค่าที่กรอก: 4 ตำแหน่ง -> 0.2545


## ข้อ 14.2 — KFold(7) ที่ไม่สับ: ค่าเฉลี่ย income_k ของ validation fold ลำดับที่ 7

- `KFold(n_splits=7)` **ไม่สับ** จึงแบ่งตามลำดับแถว: 7 ก้อนเท่า ๆ กัน (ประมาณ 172 แถว)
- **fold ลำดับที่ 7 (นับ 1) = ก้อนสุดท้าย** → `.split()` เรียงตามลำดับ จึงเป็น `folds[6]`

In [4]:
# --- ข้อ 14.2 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cust = pd.read_csv(Path('data/week14') / 'biz_customers.csv')   # ลูกค้า 1203 คน

income = cust['income_k'].to_numpy(dtype=float)
folds = list(KFold(n_splits=7).split(income))      # ไม่สับ -> เรียงตามลำดับแถว

print('ขนาดแต่ละ validation fold:')
for i, (_, va) in enumerate(folds, start=1):
    print(f'  fold {i}: {len(va)} แถว (แถว {va[0]} ถึง {va[-1]}) '
          f'| ค่าเฉลี่ย income_k = {income[va].mean():.4f}')

_, va7 = folds[6]                                  # fold ลำดับที่ 7
answer = float(income[va7].mean())

print()
print('validation fold ลำดับที่ 7: จำนวน', len(va7), 'คน')
print('ค่าเฉลี่ย income_k =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

ขนาดแต่ละ validation fold:
  fold 1: 172 แถว (แถว 0 ถึง 171) | ค่าเฉลี่ย income_k = 29.1308
  fold 2: 172 แถว (แถว 172 ถึง 343) | ค่าเฉลี่ย income_k = 30.9198
  fold 3: 172 แถว (แถว 344 ถึง 515) | ค่าเฉลี่ย income_k = 29.8698
  fold 4: 172 แถว (แถว 516 ถึง 687) | ค่าเฉลี่ย income_k = 29.5331
  fold 5: 172 แถว (แถว 688 ถึง 859) | ค่าเฉลี่ย income_k = 32.6390
  fold 6: 172 แถว (แถว 860 ถึง 1031) | ค่าเฉลี่ย income_k = 30.6634
  fold 7: 171 แถว (แถว 1032 ถึง 1202) | ค่าเฉลี่ย income_k = 28.6515

validation fold ลำดับที่ 7: จำนวน 171 คน
ค่าเฉลี่ย income_k = 28.651461988304092
ค่าที่กรอก: 4 ตำแหน่ง -> 28.6515


## ข้อ 14.3 — KFold(5) กับ Linear Regression: ค่า CV MSE

- ทำนาย `spend_k` จาก 3 ตัวแปร ด้วย `LinearRegression` (ไม่ต้อง standardize)
- แต่ละ fold: fit บน train → วัด MSE บน validation
- ค่า CV MSE = **ค่าเฉลี่ยของ 5 ค่า**

In [5]:
# --- ข้อ 14.3 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cust = pd.read_csv(Path('data/week14') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

cols = ['tenure_months', 'age', 'email_opens']
X = cust[cols].to_numpy(dtype=float)
y = cust['spend_k'].to_numpy(dtype=float)

mses = []
for i, (tr, va) in enumerate(KFold(n_splits=5).split(X), start=1):
    m = LinearRegression().fit(X[tr], y[tr])
    mse = mean_squared_error(y[va], m.predict(X[va]))
    mses.append(mse)
    print(f'  fold {i}: validation {len(va)} แถว | MSE = {mse:.6f}')

answer = float(np.mean(mses))

print()
print('CV MSE = ค่าเฉลี่ย =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

  fold 1: validation 241 แถว | MSE = 1.957707
  fold 2: validation 241 แถว | MSE = 2.280491
  fold 3: validation 241 แถว | MSE = 2.547674
  fold 4: validation 240 แถว | MSE = 2.875617
  fold 5: validation 240 แถว | MSE = 2.322320

CV MSE = ค่าเฉลี่ย = 2.3967617136101866
ค่าที่กรอก: 4 ตำแหน่ง -> 2.3968


## ข้อ 14.4 — Validation set approach: เลขคู่ = train, เลขคี่ = validation

- เป็นการแบ่งแบบกำหนดเอง ไม่ใช่ CV เพราะใช้ข้อมูลเพียงครึ่งเดียวเป็น train
- index **เริ่มที่ 0** → 0, 2, 4, … เป็น train (602 คน) และ 1, 3, 5, … เป็น validation (601 คน)
- โมเดล: พหุนามดีกรี 2 ทำนาย `basket_size` จาก `age` (ดิบ ไม่ standardize)

In [6]:
# --- ข้อ 14.4 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cust = pd.read_csv(Path('data/week14') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

idx = np.arange(len(cust))                  # เริ่มที่ 0 ตามลำดับแถว
tr = idx[idx % 2 == 0]                     # เลขคู่
va = idx[idx % 2 == 1]                     # เลขคี่

X = cust[['age']].to_numpy(dtype=float)
y = cust['basket_size'].to_numpy(dtype=float)
m = make_pipeline(PolynomialFeatures(2), LinearRegression()).fit(X[tr], y[tr])
answer = float(mean_squared_error(y[va], m.predict(X[va])))

print('train (เลขคู่) =', len(tr), 'คน | validation (เลขคี่) =', len(va), 'คน')
print('MSE บน train      =', round(float(mean_squared_error(y[tr], m.predict(X[tr]))), 6))
print('MSE บน validation =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

train (เลขคู่) = 602 คน | validation (เลขคี่) = 601 คน
MSE บน train      = 0.724544
MSE บน validation = 0.6928270875329207
ค่าที่กรอก: 4 ตำแหน่ง -> 0.6928


## ข้อ 14.5 — Leave-One-Out CV กับ KNN (K=9) บน 150 คนแรก

- `LeaveOneOut()` = ตัดลูกค้าออกทีละ 1 คนเป็น validation ใช้อีก 149 คนเป็น train
- ต้อง **fit ใหม่ทุกรอบ** รวมถึง standardize ใหม่ทุกรอบ
- ผลรวม error rate = จำนวนที่ทำนายผิด **ทั้งหมด** ÷ 150 (ไม่ใช่ค่าเฉลี่ยของ 150 ค่า!)

In [7]:
# --- ข้อ 14.5 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cust = pd.read_csv(Path('data/week14') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.neighbors import KNeighborsClassifier

sub = cust.iloc[:150]
cols = ['income_k', 'tenure_months', 'age']
X = sub[cols].to_numpy(dtype=float)
y = sub['responded'].to_numpy(dtype=int)

errors = 0
for tr, va in LeaveOneOut().split(X):
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9))
    pipe.fit(X[tr], y[tr])                    # fit ใหม่ทุกรอบ (scaler ด้วย)
    errors += int((pipe.predict(X[va]) != y[va]).sum())

answer = errors / len(y)

print('จำนวนรอบ =', len(y), '(เท่ากับจำนวนข้อมูลทุกคน)')
print('ทำนายผิด =', errors, 'คน')
print('error rate =', errors, '/', len(y), '=', answer)
print('(ถ้าสะสม error แล้วคูณ 1/150 ทีเดียว ได้ค่าเดียวกับการเฉลี่ย 150 ค่า)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนรอบ = 150 (เท่ากับจำนวนข้อมูลทุกคน)
ทำนายผิด = 43 คน
error rate = 43 / 150 = 0.2866666666666667
(ถ้าสะสม error แล้วคูณ 1/150 ทีเดียว ได้ค่าเดียวกับการเฉลี่ย 150 ค่า)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.2867


## ข้อ 14.6 — one-standard-error rule

วิธีนี้ตอบคำถาม "โมเดลซับซ้อนขึ้นได้จริงไหม" โดยไม่ต้องจ่ายค่าความแม่นยำเพิ่ม:

1. ทดลองดีกรี 1, 2, 3 แล้วได้ CV MSE ของแต่ละดีกรี (เฉลี่ย 5 fold)
2. หาดีกรีที่ CV MSE **ต่ำที่สุด** → เอา fold ทั้ง 5 ของดีกรีนั้น
3. **SE = SD ของ 5 fold (ddof=1) / √5**
4. เกณฑ์ = CV MSE ต่ำสุด + SE
5. เลือกดีกรี**ต่ำที่สุด**ที่ CV MSE ของมัน **ไม่เกินเกณฑ์** (เรียบง่ายกว่าไปดีกว่าได้ถ้าต่างกันไม่มาก)

In [8]:
# --- ข้อ 14.6 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cust = pd.read_csv(Path('data/week14') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X = cust[['age']].to_numpy(dtype=float)
y = cust['spend_k'].to_numpy(dtype=float)
folds = list(KFold(n_splits=5).split(X))

cv = {}
for d in (1, 2, 3):
    per_fold = []
    for tr, va in folds:
        m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(X[tr], y[tr])
        per_fold.append(mean_squared_error(y[va], m.predict(X[va])))
    cv[d] = np.array(per_fold)
    print(f'  ดีกรี {d}: ค่าเฉลี่ย = {cv[d].mean():.6f} | แต่ละ fold = {np.round(cv[d], 4)}')

best = min(cv, key=lambda d: cv[d].mean())
se = float(cv[best].std(ddof=1) / np.sqrt(len(cv[best])))
cutoff = float(cv[best].mean()) + se
chosen = min(d for d in (1, 2, 3) if cv[d].mean() <= cutoff)
answer = float(cv[chosen].mean())

print()
print(f'ดีกรีที่ CV MSE ต่ำสุด = {best} ({cv[best].mean():.6f})')
print(f'SE = SD(ddof=1)/sqrt(5) = {se:.6f}')
print(f'เกณฑ์ = ต่ำสุด + SE = {cutoff:.6f}')
print('-> เลือกดีกรีที่ต่ำที่สุดที่ยังไม่เกินเกณฑ์ = ดีกรี', chosen)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

  ดีกรี 1: ค่าเฉลี่ย = 2.437628 | แต่ละ fold = [2.0537 2.4138 2.57   2.8309 2.3198]
  ดีกรี 2: ค่าเฉลี่ย = 2.386927 | แต่ละ fold = [1.9986 2.4117 2.5471 2.7806 2.1967]
  ดีกรี 3: ค่าเฉลี่ย = 2.380420 | แต่ละ fold = [2.0057 2.4194 2.5252 2.7772 2.1746]

ดีกรีที่ CV MSE ต่ำสุด = 3 (2.380420)
SE = SD(ddof=1)/sqrt(5) = 0.134671
เกณฑ์ = ต่ำสุด + SE = 2.515090
-> เลือกดีกรีที่ต่ำที่สุดที่ยังไม่เกินเกณฑ์ = ดีกรี 1
ค่าที่กรอก: 4 ตำแหน่ง -> 2.4376


## ข้อ 14.7 — ช่องว่างระหว่าง validation MSE กับ train MSE

- ทั้งสองค่าเฉลี่ยจาก 5 fold เดียวกัน (ของโมเดลพหุนามดีกรี 3 ทำนาย `spend_k` จาก `age`)
- คำตอบ = **ค่าเฉลี่ย validation MSE − ค่าเฉลี่ย train MSE**
- ตัวเลขนี้คือ "ช่องว่างการทั่วไป (generalization gap)"
  - **ใกล้ 0** = โมเดลไม่ได้จำซ้ำ (ไม่ overfit)
  - **ใหญ่มาก** = จำซ้ำ (overfit) → ควรลดความซับซ้อนลง

In [9]:
# --- ข้อ 14.7 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cust = pd.read_csv(Path('data/week14') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X = cust[['age']].to_numpy(dtype=float)
y = cust['spend_k'].to_numpy(dtype=float)

va_mse, tr_mse = [], []
for tr, va in KFold(n_splits=5).split(X):
    m = make_pipeline(PolynomialFeatures(3), LinearRegression()).fit(X[tr], y[tr])
    va_mse.append(mean_squared_error(y[va], m.predict(X[va])))
    tr_mse.append(mean_squared_error(y[tr], m.predict(X[tr])))

answer = float(np.mean(va_mse) - np.mean(tr_mse))

print('MSE บน train ของแต่ละ fold     =', np.round(tr_mse, 4))
print('MSE บน validation ของแต่ละ fold =', np.round(va_mse, 4))
print()
print('ค่าเฉลี่ย train      =', round(float(np.mean(tr_mse)), 6))
print('ค่าเฉลี่ย validation =', round(float(np.mean(va_mse)), 6))
print('ช่องว่าง (ค่าตอบ) =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

MSE บน train ของแต่ละ fold     = [2.4624 2.3584 2.333  2.2685 2.4212]
MSE บน validation ของแต่ละ fold = [2.0057 2.4194 2.5252 2.7772 2.1746]

ค่าเฉลี่ย train      = 2.368685
ค่าเฉลี่ย validation = 2.38042
ช่องว่าง (ค่าตอบ) = 0.011734477654913977
ค่าที่กรอก: 4 ตำแหน่ง -> 0.0117


## ข้อ 14.8 — Stratified 5-Fold CV ของ LDA (ทำนายการเลิกซื้อ)

- คลาสเลิกซื้อมีน้อย (~12%) ถ้าใช้ `KFold` ธรรมดา แต่ละ fold จะมีสัดส่วนผู้เลิกซื้อต่างกันมาก
  → ใช้ **`StratifiedKFold`** เพื่อให้ทุก fold มีสัดส่วนใกล้เคียงกัน
- คะแนนของแต่ละคนคือ **ความน่าจะเป็นของคลาส 1**
- ค่า CV AUC = ค่าเฉลี่ยของ AUC ทั้ง 5 fold

In [10]:
# --- ข้อ 14.8 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cust = pd.read_csv(Path('data/week14') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import roc_auc_score

cols = ['tenure_months', 'basket_size', 'income_k']
X = cust[cols].to_numpy(dtype=float)
y = cust['churn'].to_numpy(dtype=int)
print('สัดส่วน churn = 1 ทั้งไฟล์ =', round(float(y.mean()), 4))

aucs = []
for i, (tr, va) in enumerate(StratifiedKFold(n_splits=5).split(X, y), start=1):
    m = LinearDiscriminantAnalysis().fit(X[tr], y[tr])
    auc = roc_auc_score(y[va], m.predict_proba(X[va])[:, 1])
    aucs.append(auc)
    print(f'  fold {i}: สัดส่วน churn ใน train = {y[tr].mean():.4f} | ใน validation = {y[va].mean():.4f} | AUC = {auc:.6f}')

answer = float(np.mean(aucs))

print()
print('เฉลี่ย 5 fold =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

สัดส่วน churn = 1 ทั้งไฟล์ = 0.1189
  fold 1: สัดส่วน churn ใน train = 0.1185 | ใน validation = 0.1203 | AUC = 0.728530
  fold 2: สัดส่วน churn ใน train = 0.1185 | ใน validation = 0.1203 | AUC = 0.577424
  fold 3: สัดส่วน churn ใน train = 0.1185 | ใน validation = 0.1203 | AUC = 0.702830
  fold 4: สัดส่วน churn ใน train = 0.1194 | ใน validation = 0.1167 | AUC = 0.668632
  fold 5: สัดส่วน churn ใน train = 0.1194 | ใน validation = 0.1167 | AUC = 0.720182

เฉลี่ย 5 fold = 0.6795194720698949
ค่าที่กรอก: 4 ตำแหน่ง -> 0.6795


## ข้อ 14.9 — KFold(5) กับ KNN Regression (K=15)

- `KNeighborsRegressor` ทำนายค่าเฉลี่ยของ y ของเพื่อนบ้าน K คน
- standardize ใหม่ในทุก fold (pipeline)
- ค่า CV MSE = ค่าเฉลี่ยของ 5 fold

In [11]:
# --- ข้อ 14.9 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cust = pd.read_csv(Path('data/week14') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

cols = ['spend_k', 'income_k', 'tenure_months']
X = cust[cols].to_numpy(dtype=float)
y = cust['basket_size'].to_numpy(dtype=float)

mses = []
for i, (tr, va) in enumerate(KFold(n_splits=5).split(X), start=1):
    pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=15))
    pipe.fit(X[tr], y[tr])
    mse = mean_squared_error(y[va], pipe.predict(X[va]))
    mses.append(mse)
    print(f'  fold {i}: validation {len(va)} แถว | MSE = {mse:.6f}')

answer = float(np.mean(mses))

print()
print('CV MSE = ค่าเฉลี่ย =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

  fold 1: validation 241 แถว | MSE = 0.644564
  fold 2: validation 241 แถว | MSE = 0.666460
  fold 3: validation 241 แถว | MSE = 0.642595
  fold 4: validation 240 แถว | MSE = 0.720231
  fold 5: validation 240 แถว | MSE = 0.611245

CV MSE = ค่าเฉลี่ย = 0.6570190213616106
ค่าที่กรอก: 4 ตำแหน่ง -> 0.6570


## ข้อ 14.10 — เทียบ CV MSE กับ Test MSE ของการแบ่งครั้งเดียว

**วิธี A (single split):** train = 960 แถวแรก, test = 243 แถวที่เหลือ → Test MSE
**วิธี B (cross-validation):** `KFold(n_splits=5)` ทั้ง 1203 แถว → ค่าเฉลี่ย validation MSE

คำตอบ = วิธี B − วิธี A (**ติดลบได้**)

- ถ้าเป็นบวก → การแบ่งครั้งเดียวทำให้เห็นภาพที่ดีเกินจริง (ข้อมูลที่แบ่งแยกอาจง่ายกว่าทั้งชุด)
- ถ้าเป็นลบ → การแบ่งครั้งเดียวทำให้เห็นภาพที่แย่กว่าจริง

In [12]:
# --- ข้อ 14.10 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

cust = pd.read_csv(Path('data/week14') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

cols = ['income_k', 'basket_size', 'web_visits']
X = cust[cols].to_numpy(dtype=float)
y = cust['spend_k'].to_numpy(dtype=float)

# วิธี A: แบ่งครั้งเดียวตามตำแหน่งแถว
m = LinearRegression().fit(X[:960], y[:960])
test_mse = float(mean_squared_error(y[960:], m.predict(X[960:])))

# วิธี B: K-Fold ทั้งไฟล์
cv = []
for tr, va in KFold(n_splits=5).split(X):
    mm = LinearRegression().fit(X[tr], y[tr])
    cv.append(mean_squared_error(y[va], mm.predict(X[va])))
cv_mse = float(np.mean(cv))

answer = cv_mse - test_mse

print('วิธี A (960/243): Test MSE =', round(test_mse, 6))
print('วิธี B (KFold 5) : CV MSE   =', round(cv_mse, 6), '(fold =', np.round(cv, 4), ')')
print('ผลต่าง B - A =', answer, '->', 'CV สูงกว่า = การแบ่งครั้งเดียวทำให้เห็นภาพที่ดีเกินจริง'
      if answer > 0 else '-> CV ต่ำกว่า = การแบ่งครั้งเดียวทำให้เห็นภาพที่แย่กว่า')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

วิธี A (960/243): Test MSE =

 1.761803
วิธี B (KFold 5) : CV MSE   = 1.778932 (fold = [1.7743 1.6438 1.8678 1.8524 1.7565] )
ผลต่าง B - A = 0.01712913231679436 -> CV สูงกว่า = การแบ่งครั้งเดียวทำให้เห็นภาพที่ดีเกินจริง
ค่าที่กรอก: 4 ตำแหน่ง -> 0.0171


## สรุปคำตอบทั้ง 10 ข้อ (Week 14)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อนที่เว็บยอมรับ`

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 14.1 | ค่าเฉลี่ย F1 จาก Stratified 5-fold (KNN K=11) | **0.2545** | 4 ตำแหน่ง | 0.00015 |
| 14.2 | ค่าเฉลี่ย income_k ของ validation fold ที่ 7 | **28.6515** | 4 ตำแหน่ง | 0.00015 |
| 14.3 | CV MSE ของ LinearRegression (KFold 5) | **2.3968** | 4 ตำแหน่ง | 0.00015 |
| 14.4 | MSE บน validation (train=เลขคู่, val=เลขคี่) | **0.6928** | 4 ตำแหน่ง | 0.00015 |
| 14.5 | LOOCV error rate ของ KNN K=9 บน 150 คน | **0.2867** | 4 ตำแหน่ง | 0.00015 |
| 14.6 | CV MSE ของดีกรีที่เลือกด้วย 1-SE rule | **2.4376** | 4 ตำแหน่ง | 0.00015 |
| 14.7 | ค่าเฉลี่ย val MSE - ค่าเฉลี่ย train MSE | **0.0117** | 4 ตำแหน่ง | 0.00015 |
| 14.8 | ค่าเฉลี่ย AUC ของ LDA (Stratified 5-fold) | **0.6795** | 4 ตำแหน่ง | 0.00015 |
| 14.9 | CV MSE ของ KNN Regression K=15 | **0.6570** | 4 ตำแหน่ง | 0.00015 |
| 14.10 | CV MSE - Test MSE (960/243) | **0.0171** | 4 ตำแหน่ง | 0.00015 |

### สรุปทั้งเทอม

- **14.1** KNN classifier ได้ F1 ต่ำมาก (0.2545) เพราะข้อมูลไม่มีโครงสร้างที่เพื่อนบ้านจะช่วยจำแนกได้
- **14.5** LOOCV บน 150 คนทำนายผิด 43 คน = 28.67%
- **14.6** one-standard-error rule เลือก **ดีกรี 1** ทั้งที่ดีกรี 3 ดีที่สุด
  เพราะ SE = 0.1347 ทำให้ดีกรี 1, 2, 3 ถือว่า "ไม่ต่างกันอย่างมีนัยสำคัญ" → เลือกอันง่ายที่สุด
- **14.7** ช่องว่าง train/validation เพียง 0.0117 → โมเดลดีกรี 3 **ไม่ได้จำซ้ำ** มากนัก
- **14.10** CV MSE สูงกว่า Test MSE เล็กน้อย (0.0171) → การแบ่งครั้งเดียวให้ภาพที่ดีกว่าจริงเล็กน้อย

🎉 **จบครบทั้ง 14 สัปดาห์ / 140 ข้อ**

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน